In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import sys
sys.path.insert(0, '../')

# Define the target directory
target_directory = '/content/drive/MyDrive/Colab Notebooks/BEHRT'

# Change the current working directory
os.chdir(target_directory)

print(f"Current working directory changed to: {os.getcwd()}")

In [ ]:
!pip install pytorch_pretrained_bert
!pip install duckdb

In [ ]:
from common.common import create_folder
from common.pytorch import load_model
import pytorch_pretrained_bert as Bert
from model.utils import age_vocab
from common.common import load_obj
from dataLoader.MLM import MLMLoader
from torch.utils.data import DataLoader
import pandas as pd
from model.MLM import BertForMaskedLM
from model.optimiser import adam
import sklearn.metrics as skm
import numpy as np
import torch
import time
import torch.nn as nn
import os
import duckdb

def read_parquet_safe(path):
    """pd.read_parquet() / pyarrow's to_pandas() / to_pylist() can all throw
    'ArrowNotImplementedError: Nested data conversions not implemented for
    chunked array outputs' on parquet files with very large list-type
    columns (our 'code'/'age' sequences, now dominated by vitals -- some
    patients have thousands of readings). This is a known PyArrow limitation
    with big nested/list columns, not something fixable by combining chunks
    or changing the conversion call -- both still go through the same
    internal PyArrow code path.
    DuckDB uses a completely different (and more robust) engine to read
    parquet and convert to pandas, so it sidesteps the bug entirely."""
    con = duckdb.connect()
    return con.execute(f"SELECT * FROM read_parquet('{path}')").df()


In [ ]:
import pandas as pd
import pickle
import duckdb

print("Creating the BEHRT dictionary...")

# IMPORTANT: this used to load the WHOLE parquet into a pandas DataFrame just
# to build a set of unique tokens -- that's what was blowing up RAM. DuckDB's
# UNNEST pulls the distinct tokens directly out of the list column via SQL,
# without ever materializing the full nested DataFrame in Python.
con = duckdb.connect()
unique_tokens = set(
    row[0] for row in con.execute(
        "SELECT DISTINCT UNNEST(code) FROM read_parquet('0data/behrt_pretrain_data.parquet')"
    ).fetchall()
)

# IMPORTANT: this repo's dataLoader/utils.py and dataLoader/MLM.py use
# UNBRACKETED special token names ('PAD', 'UNK', 'CLS', 'SEP', 'MASK') --
# NOT the HuggingFace-style '[PAD]', '[CLS]' etc. Using the bracketed style
# breaks MLMLoader (token2idx['PAD'], token2idx["MASK"], etc. would all
# KeyError). 'SEP' is already present in unique_tokens (it's literally in
# the code sequences as the visit separator) so we exclude it from
# special_tokens to avoid it getting silently overwritten to a different
# index by the dict comprehension.
special_tokens = ['PAD', 'UNK', 'CLS', 'SEP', 'MASK']
combined_tokens = special_tokens + [t for t in unique_tokens if t not in special_tokens]
token2idx = {token: i for i, token in enumerate(combined_tokens)}
idx2token = {i: token for token, i in token2idx.items()}

# IMPORTANT: MLMLoader/BEHRT expect BertVocab to be a dict WRAPPING
# token2idx/idx2token (BertVocab['token2idx']), not a flat token->idx dict.
vocab = {'token2idx': token2idx, 'idx2token': idx2token}

# Save the dictionary as a pickle file
with open('0data/vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)

print(f"Success! Your vocabulary has {len(token2idx)} unique medical words and is saved in 0data/vocab.pkl.")


In [ ]:
class BertConfig(Bert.modeling.BertConfig):
    def __init__(self, config):
        super(BertConfig, self).__init__(
            vocab_size_or_config_json_file=config.get('vocab_size'),
            hidden_size=config['hidden_size'],
            num_hidden_layers=config.get('num_hidden_layers'),
            num_attention_heads=config.get('num_attention_heads'),
            intermediate_size=config.get('intermediate_size'),
            hidden_act=config.get('hidden_act'),
            hidden_dropout_prob=config.get('hidden_dropout_prob'),
            attention_probs_dropout_prob=config.get('attention_probs_dropout_prob'),
            max_position_embeddings = config.get('max_position_embedding'),
            initializer_range=config.get('initializer_range'),
        )
        self.seg_vocab_size = config.get('seg_vocab_size')
        self.age_vocab_size = config.get('age_vocab_size')

class TrainConfig(object):
    def __init__(self, config):
        self.batch_size = config.get('batch_size')
        self.use_cuda = config.get('use_cuda')
        self.max_len_seq = config.get('max_len_seq')
        self.train_loader_workers = config.get('train_loader_workers')
        self.test_loader_workers = config.get('test_loader_workers')
        self.device = config.get('device')
        self.output_dir = config.get('output_dir')
        self.output_name = config.get('output_name')
        self.best_name = config.get('best_name')

In [ ]:
file_config = {
    'vocab':'0data/vocab.pkl',  # vocabulary idx2token, token2idx
    'data': '0data/behrt_pretrain_data.parquet',  # formated data
    'model_path': 'saved_models/', # where to save model
    'model_name': 'behrt_pretrain_model.pth', # model name
    'file_name': 'mlm_training_log.txt',  # log path
}
create_folder(file_config['model_path'])

In [ ]:
global_params = {
    'max_seq_len': 300,  # antes 64 -- debe coincidir con MAX_SEQ_LENGTH del fine-tuning (300), si no el position_embeddings pre-entrenado (64 posiciones) no encaja con el del fine-tuning (config max_position_embeddings=512) y load_state_dict revienta por size mismatch
    'max_age': 110,
    'month': 1,
    'age_symbol': None,
    'min_visit': 5,
    'gradient_accumulation_steps': 8
}

optim_param = {
    'lr': 3e-5,
    'warmup_proportion': 0.1,
    'weight_decay': 0.01
}

train_params = {
    'batch_size': 32,
    'use_cuda': True,
    'max_len_seq': global_params['max_seq_len'],
    'device': 'cuda:0'
}

In [ ]:
import pickle

with open(file_config['vocab'], 'rb') as f:
    BertVocab = pickle.load(f)

ageVocab, _ = age_vocab(max_age=global_params['max_age'], mon=global_params['month'], symbol=global_params['age_symbol'])


In [ ]:
data = read_parquet_safe(file_config['data'])
n_before = len(data)
# remove patients with visits less than min visit
data['length'] = data['code'].apply(lambda x: len([i for i in range(len(x)) if x[i] == 'SEP']))
data = data[data['length'] >= global_params['min_visit']]
print(f'Patients before min_visit filter: {n_before} | after: {len(data)} (dropped {n_before - len(data)})')
data = data.reset_index(drop=True)

steps_per_epoch = len(data) // train_params['batch_size']
print(f"Steps per epoch (batch_size={train_params['batch_size']}): {steps_per_epoch}")

In [ ]:
Dset = MLMLoader(data, BertVocab['token2idx'], ageVocab, max_len=train_params['max_len_seq'], code='code')
trainload = DataLoader(dataset=Dset, batch_size=train_params['batch_size'], shuffle=True, num_workers=0)

In [ ]:
model_config = {
    'vocab_size': len(BertVocab['token2idx'].keys()), # number of disease + symbols for word embedding
    'hidden_size': 288, # word embedding and seg embedding hidden size
    'seg_vocab_size': 2, # number of vocab for seg embedding
    'age_vocab_size': len(ageVocab.keys()), # number of vocab for age embedding
    'max_position_embedding': train_params['max_len_seq'], # maximum number of tokens
    'hidden_dropout_prob': 0.1, # dropout rate
    'num_hidden_layers': 6, # number of multi-head attention layers required
    'num_attention_heads': 12, # number of attention heads
    'attention_probs_dropout_prob': 0.1, # multi-head attention dropout rate
    'intermediate_size': 512, # the size of the "intermediate" layer in the transformer encoder
    'hidden_act': 'gelu', # The non-linear activation function in the encoder and the pooler "gelu", 'relu', 'swish' are supported
    'initializer_range': 0.02, # parameter weight initializer range
}

In [ ]:
conf = BertConfig(model_config)
model = BertForMaskedLM(conf)

In [ ]:
model = model.to(train_params['device'])
optim = adam(params=list(model.named_parameters()), config=optim_param)

In [ ]:
def cal_acc(label, pred):
    logs = nn.LogSoftmax()
    label=label.cpu().numpy()
    ind = np.where(label!=-1)[0]
    truepred = pred.detach().cpu().numpy()
    truepred = truepred[ind]
    truelabel = label[ind]
    truepred = logs(torch.tensor(truepred))
    outs = [np.argmax(pred_x) for pred_x in truepred.numpy()]
    precision = skm.precision_score(truelabel, outs, average='micro')
    return precision

In [ ]:
STEP_LOG_PATH = os.path.join(file_config['model_path'], 'mlm_step_log.txt')
if not os.path.exists(STEP_LOG_PATH):
    with open(STEP_LOG_PATH, 'w') as slog:
        slog.write('epoch\tcnt\tglobal_step\tloss\tprecision\ttime\n')

def train(e, loader):
    tr_loss = 0
    temp_loss = 0
    nb_tr_examples, nb_tr_steps = 0, 0
    cnt = 0
    start = time.time()
    epoch_precisions = []

    for step, batch in enumerate(loader):
        cnt += 1
        batch = tuple(t.to(train_params['device']) for t in batch)
        age_ids, input_ids, posi_ids, segment_ids, attMask, masked_label = batch
        loss, pred, label = model(input_ids, age_ids, segment_ids, posi_ids, attention_mask=attMask, masked_lm_labels=masked_label)
        if global_params['gradient_accumulation_steps'] > 1:
            loss = loss / global_params['gradient_accumulation_steps']
        loss.backward()

        temp_loss += loss.item()
        tr_loss += loss.item()

        nb_tr_examples += input_ids.size(0)
        nb_tr_steps += 1

        if step % 200 == 0:
            step_precision = cal_acc(label, pred)
            epoch_precisions.append(step_precision)
            step_loss = temp_loss / 2000
            elapsed = time.time() - start
            print("epoch: {}\t| cnt: {}\t|Loss: {}\t| precision: {:.4f}\t| time: {:.2f}".format(e, cnt, step_loss, step_precision, elapsed))

            # Global step consistente incluso entre reinicios/resumes, porque se
            # calcula a partir de steps_per_epoch (celda 7), no de un contador
            # frágil que dependa de detectar "saltos" de época.
            global_step = e * steps_per_epoch + cnt
            with open(STEP_LOG_PATH, 'a') as slog:
                slog.write(f'{e}\t{cnt}\t{global_step}\t{step_loss}\t{step_precision}\t{elapsed}\n')
                slog.flush()
                os.fsync(slog.fileno())

            temp_loss = 0
            start = time.time()

        if (step + 1) % global_params['gradient_accumulation_steps'] == 0:
            optim.step()
            optim.zero_grad()

    avg_precision = float(np.mean(epoch_precisions)) if epoch_precisions else float('nan')

    print("** ** * Saving pre-trained model (latest + per-epoch checkpoint) ** ** * ")
    model_to_save = model.module if hasattr(model, 'module') else model
    create_folder(file_config['model_path'])

    output_model_file = os.path.join(file_config['model_path'], file_config['model_name'])
    torch.save(model_to_save.state_dict(), output_model_file)

    epoch_ckpt_name = file_config['model_name'].replace('.pth', f'_epoch_{e}.pth')
    epoch_ckpt_path = os.path.join(file_config['model_path'], epoch_ckpt_name)
    torch.save(model_to_save.state_dict(), epoch_ckpt_path)

    cost = time.time() - start
    return tr_loss, cost, avg_precision

In [20]:
from tqdm.auto import tqdm
import glob

TARGET_EPOCHS = 10

existing_ckpts = glob.glob(os.path.join(file_config['model_path'], 'behrt_pretrain_model_epoch_*.pth'))
if existing_ckpts:
    completed_epochs = sorted(int(p.split('_epoch_')[-1].replace('.pth', '')) for p in existing_ckpts)
    last_epoch = completed_epochs[-1]
    start_epoch = last_epoch + 1

    model_to_load = model.module if hasattr(model, 'module') else model
    state_dict = torch.load(
        os.path.join(file_config['model_path'], f'behrt_pretrain_model_epoch_{last_epoch}.pth'),
        map_location=train_params['device']
    )
    model_to_load.load_state_dict(state_dict)
    print(f"Checkpoint de epoch {last_epoch} encontrado y cargado. Retomando desde epoch {start_epoch}.")
else:
    start_epoch = 0
    print("No hay checkpoints previos. Empezando desde epoch 0.")

log_path = os.path.join(file_config['model_path'], file_config['file_name'])
write_header = not os.path.exists(log_path)
f = open(log_path, "a")
if write_header:
    f.write('{}\t{}\t{}\t{}\n'.format('epoch', 'loss', 'precision', 'time'))
    f.flush()
    os.fsync(f.fileno())  # fuerza la sincronización real con Drive, no solo el buffer de Python

data_len = len(data)

for e in tqdm(range(start_epoch, TARGET_EPOCHS), desc="MLM pretraining epochs"):
    loss, time_cost, avg_precision = train(e, trainload)
    loss = loss / data_len
    f.write('{}\t{}\t{}\t{}\n'.format(e, loss, avg_precision, time_cost))
    f.flush()
    os.fsync(f.fileno())
f.close()

Checkpoint de epoch 9 encontrado y cargado. Retomando desde epoch 10.


MLM pretraining epochs:   0%|          | 0/1 [00:00<?, ?it/s]

epoch: 10	| cnt: 1	|Loss: 0.0006419422626495362	| precision: 0.0690	| time: 2.44


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

step_log = pd.read_csv(os.path.join(file_config['model_path'], 'mlm_step_log.txt'), sep='\t')
print(f"{len(step_log)} puntos registrados (cada 200 steps)")

def smooth_curve(data, weight=0.90):
    smoothed = []
    last = data[0]
    for point in data:
        smooth_val = last * weight + (1 - weight) * point
        smoothed.append(smooth_val)
        last = smooth_val
    return smoothed

smoothed_loss = smooth_curve(step_log['loss'].tolist(), weight=0.90)
smoothed_prec = smooth_curve(step_log['precision'].tolist(), weight=0.90)

# --- Loss ---
plt.figure(figsize=(10, 5))
plt.plot(step_log['global_step'], step_log['loss'], color='red', alpha=0.2, label='Raw')
plt.plot(step_log['global_step'], smoothed_loss, color='red', linewidth=2, label='Loss (smoothed)')
plt.title('Learning Curve: Loss', fontsize=14, fontweight='bold')
plt.xlabel('Global Training Steps', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
loss_path = os.path.join(file_config['model_path'], 'mlm_loss_curve.png')
plt.savefig(loss_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Guardado: {loss_path}")

# --- Precision ---
plt.figure(figsize=(10, 5))
plt.plot(step_log['global_step'], step_log['precision'], color='blue', alpha=0.2, label='Raw')
plt.plot(step_log['global_step'], smoothed_prec, color='blue', linewidth=2, label='Precision (smoothed)')
plt.title('Learning Curve: Precision', fontsize=14, fontweight='bold')
plt.xlabel('Global Training Steps', fontsize=12)
plt.ylabel('Precision (0 to 1)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
prec_path = os.path.join(file_config['model_path'], 'mlm_precision_curve.png')
plt.savefig(prec_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Guardado: {prec_path}")